In [1]:
!pip install earthengine-api geemap geopandas pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 6.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 10.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]


In [3]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [7]:

# ============================================================
# PANAMA ONE HEALTH DISTRICT DATAFRAME
# ANATI 2025 district boundaries
#
# Outcome placeholder: MODIS land surface temperature
# Predictors: WaPOR AETI, SINIA land cover, WDPA protected areas
# Supporting variable: GHSL population (2020)
#
# VIIRS nighttime radiance and Sentinel-5P NO2 are excluded.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import ee
import requests
import pandas as pd
import geopandas as gpd

from shapely.geometry import shape
from pathlib import Path


# ------------------------------------------------------------
# 2. INITIALISE GOOGLE EARTH ENGINE
# ------------------------------------------------------------

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine connected.")


# ------------------------------------------------------------
# 3. DOWNLOAD ANATI 2025 DISTRICT BOUNDARIES
# ------------------------------------------------------------

url = (
    "https://sigigntg.anati.gob.pa/arcgisserver/rest/services/"
    "Mapa_Web_Divisi%C3%B3n_Pol%C3%ADtica_Administrativa_2025a_MIL1/"
    "MapServer/7/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

geojson = response.json()
features = geojson["features"]

if not features:
    raise ValueError("ANATI returned no district features.")

attributes = []
geometries = []

for feature in features:
    properties = feature["properties"]
    geometry = feature["geometry"]

    if geometry is None:
        continue

    attributes.append(properties)
    geometries.append(shape(geometry))

districts = gpd.GeoDataFrame(
    pd.DataFrame(attributes),
    geometry=gpd.GeoSeries(geometries, crs="EPSG:4326"),
    crs="EPSG:4326"
)

# Check that the expected ANATI fields exist.
required_fields = ["nomb_dist", "nomb_prov", "cod_dist"]

missing_fields = [
    field for field in required_fields
    if field not in districts.columns
]

if missing_fields:
    raise ValueError(
        f"Missing expected ANATI fields: {missing_fields}"
    )

# Standardise names.
districts = districts.rename(
    columns={
        "nomb_dist": "DISTRICT",
        "nomb_prov": "PROVINCE",
        "cod_dist": "DISTRICT_CODE"
    }
)

districts["DISTRICT"] = districts["DISTRICT"].astype(str).str.strip()
districts["PROVINCE"] = districts["PROVINCE"].astype(str).str.strip()
districts["DISTRICT_CODE"] = districts["DISTRICT_CODE"].astype(str)

# Repair invalid geometries.
districts["geometry"] = districts.geometry.make_valid()

# Dissolve polygon fragments belonging to the same district.
districts = districts.dissolve(
    by=["DISTRICT_CODE", "PROVINCE", "DISTRICT"],
    as_index=False
)

# Create a consistent district key for joining datasets.
districts["ADMIN_KEY"] = (
    districts["PROVINCE"].str.upper()
    + "|"
    + districts["DISTRICT"].str.upper()
)

if districts["ADMIN_KEY"].duplicated().any():
    raise ValueError("Duplicate district keys found after dissolving.")

print("ANATI districts:", len(districts))


# ------------------------------------------------------------
# 4. CONVERT DISTRICTS TO EARTH ENGINE
# ------------------------------------------------------------

districts_ee = ee.FeatureCollection(
    districts.__geo_interface__
)

panama = districts_ee.geometry()

print("District boundaries ready for analysis.")


# ------------------------------------------------------------
# 5. HELPER FUNCTION: REDUCE AN IMAGE BY DISTRICT
# ------------------------------------------------------------

def district_reduce(image, reducer, scale, output_name):

    image = image.rename("value")

    reducer = reducer.setOutputs([output_name])

    reduced = image.reduceRegions(
        collection=districts_ee,
        reducer=reducer,
        scale=scale,
        tileScale=4
    )

    result = reduced.getInfo()

    rows = [
        feature["properties"]
        for feature in result["features"]
    ]

    dataframe = pd.DataFrame(rows)

    if output_name not in dataframe.columns:
        raise ValueError(
            f"Expected output '{output_name}' was not returned. "
            f"Returned columns: {list(dataframe.columns)}"
        )

    return dataframe[["ADMIN_KEY", output_name]]


# ------------------------------------------------------------
# 6. MODIS LAND SURFACE TEMPERATURE
# PLACEHOLDER OUTCOME VARIABLE
# ------------------------------------------------------------

print("Calculating MODIS land surface temperature...")

def mask_modis_lst(image):

    qc = image.select("QC_Day")

    good_quality = qc.bitwiseAnd(3).eq(0)

    lst = (
        image.select("LST_Day_1km")
        .updateMask(good_quality)
    )

    return lst.copyProperties(
        image,
        ["system:time_start"]
    )


modis = (
    ee.ImageCollection("MODIS/061/MOD11A2")
    .filterDate("2021-01-01", "2024-01-01")
    .filterBounds(panama)
    .map(mask_modis_lst)
)

modis_count = modis.size().getInfo()

print("MODIS images:", modis_count)

if modis_count == 0:
    raise ValueError("No MODIS images found for the selected period.")

lst_image = (
    modis.mean()
    .multiply(0.02)
    .subtract(273.15)
    .rename("value")
)

lst_df = district_reduce(
    lst_image,
    ee.Reducer.mean(),
    1000,
    "LST_mean_C"
)


# ------------------------------------------------------------
# 7. WaPOR ACTUAL EVAPOTRANSPIRATION AND INTERCEPTION
# ------------------------------------------------------------

print("Checking WaPOR data availability...")

# Test the two possible WaPOR collection IDs.
wapor_candidates = [
    "FAO/WAPOR/3/L1_AETI_D",
    "projects/UNFAO/wapor/v3/L1-AETI-D"
]

wapor = None
wapor_band = None

for collection_id in wapor_candidates:

    try:
        candidate = ee.ImageCollection(collection_id)

        total_images = candidate.size().getInfo()

        if total_images == 0:
            continue

        bands = candidate.first().bandNames().getInfo()

        print("WaPOR collection:", collection_id)
        print("Available bands:", bands)

        # Use the expected AETI band if it exists.
        if "L1-AETI-D" in bands:
            band = "L1-AETI-D"
        elif len(bands) == 1:
            band = bands[0]
        else:
            continue

        filtered = (
            candidate
            .filterDate("2021-01-01", "2024-01-01")
            .filterBounds(panama)
            .select(band)
        )

        filtered_count = filtered.size().getInfo()

        print("WaPOR images for Panama and selected dates:", filtered_count)

        if filtered_count > 0:
            wapor = filtered
            wapor_band = band
            break

    except Exception as error:
        print(f"Could not use {collection_id}: {error}")


if wapor is None:
    raise ValueError(
        "No usable WaPOR collection was found. "
        "Check the collection IDs, band names and Earth Engine access."
    )

# WaPOR AETI-D scale factor: 0.1.
# This is the mean daily AETI across the available dekadal images,
# not an annual accumulated total.
aeti_image = (
    wapor.mean()
    .multiply(0.1)
    .rename("value")
)

aeti_df = district_reduce(
    aeti_image,
    ee.Reducer.mean(),
    250,
    "AETI_mean"
)


# ------------------------------------------------------------
# 8. SINIA LAND COVER
# ------------------------------------------------------------

print("Calculating dominant land-cover class...")

landcover = ee.Image(
    "projects/deforestation-panama/assets/landuse_2021"
)

# Reclassify the original SINIA land-cover codes.
from_list = list(range(1, 34))

to_list = [
    10, 20, 80, 80, 80, 80, 20, 20, 60, 60, 60,
    70, 70, 50, 40, 50, 40, 40, 40, 50, 40, 40,
    40, 40, 50, 60, 27, 28, 29, 70, 70, 70, 70
]

landcover_reclassified = (
    landcover
    .remap(from_list, to_list)
    .rename("value")
)

landcover_df = district_reduce(
    landcover_reclassified,
    ee.Reducer.mode(),
    30,
    "dominant_landcover"
)


# ------------------------------------------------------------
# 9. PROTECTED AREA COVERAGE
# ------------------------------------------------------------

print("Calculating protected-area coverage...")

wdpa = (
    ee.FeatureCollection("WCMC/WDPA/current/polygons")
    .filter(ee.Filter.eq("ISO3", "PAN"))
    .filterBounds(panama)
)

wdpa_count = wdpa.size().getInfo()

print("WDPA protected-area features:", wdpa_count)

if wdpa_count == 0:
    raise ValueError("No WDPA protected areas found for Panama.")

# Rasterise protected areas as 1 and other areas as 0.
protected_raster = (
    ee.Image.constant(0)
    .byte()
    .paint(wdpa, 1)
    .rename("value")
    .reproject(
        crs="EPSG:6933",
        scale=1000
    )
)

# Mean of the 0/1 raster multiplied by 100 gives
# approximate protected-area coverage percentage.
protected_df = district_reduce(
    protected_raster,
    ee.Reducer.mean(),
    1000,
    "protected_area_fraction"
)

protected_df["protected_area_pct"] = (
    protected_df["protected_area_fraction"] * 100
)

protected_df = protected_df[
    ["ADMIN_KEY", "protected_area_pct"]
]


# ------------------------------------------------------------
# 10. GHSL POPULATION (2020)
# SUPPORTING VARIABLE
# ------------------------------------------------------------

print("Calculating district population...")

population = ee.Image(
    "JRC/GHSL/P2023A/GHS_POP/2020"
).select("population_count")

population_df = district_reduce(
    population,
    ee.Reducer.sum(),
    100,
    "population_2020"
)


# ------------------------------------------------------------
# 11. COMBINE ALL VARIABLES
# ------------------------------------------------------------

print("Combining district indicators...")

final_df = districts[
    [
        "ADMIN_KEY",
        "DISTRICT",
        "PROVINCE",
        "DISTRICT_CODE"
    ]
].copy()

dataframes = [
    lst_df,
    aeti_df,
    landcover_df,
    protected_df,
    population_df
]

for dataframe in dataframes:

    final_df = final_df.merge(
        dataframe,
        on="ADMIN_KEY",
        how="left",
        validate="one_to_one"
    )

numeric_columns = [
    "LST_mean_C",
    "AETI_mean",
    "dominant_landcover",
    "protected_area_pct",
    "population_2020"
]

for column in numeric_columns:

    final_df[column] = pd.to_numeric(
        final_df[column],
        errors="coerce"
    )


# ------------------------------------------------------------
# 12. QUALITY CHECKS
# ------------------------------------------------------------

print("\nFinal dataframe preview:")
print(final_df.head())

print("\nNumber of districts:", len(final_df))

print("\nMissing values:")
print(final_df.isna().sum())

print("\nDataframe dimensions:", final_df.shape)

if len(final_df) != len(districts):
    raise ValueError(
        "The final dataframe does not contain one row per ANATI district."
    )


# ------------------------------------------------------------
# 13. EXPORT CSV AND GEOPACKAGE
# ------------------------------------------------------------

output_folder = Path("panama_one_health_output")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# Export CSV.
csv_path = output_folder / "panama_one_health_simplified.csv"

final_df.to_csv(
    csv_path,
    index=False
)

# Join the indicators back to the district geometries.
final_gdf = districts[
    [
        "ADMIN_KEY",
        "DISTRICT",
        "PROVINCE",
        "DISTRICT_CODE",
        "geometry"
    ]
].merge(
    final_df.drop(
        columns=[
            "DISTRICT",
            "PROVINCE",
            "DISTRICT_CODE"
        ]
    ),
    on="ADMIN_KEY",
    how="left",
    validate="one_to_one"
)

final_gdf = gpd.GeoDataFrame(
    final_gdf,
    geometry="geometry",
    crs="EPSG:4326"
)

# Export GeoPackage.
gpkg_path = output_folder / "panama_one_health_simplified.gpkg"

final_gdf.to_file(
    gpkg_path,
    layer="district_data",
    driver="GPKG"
)

print("\nExport complete.")
print("CSV:", csv_path)
print("GeoPackage:", gpkg_path)


Google Earth Engine connected.


ValueError: Missing expected ANATI fields: ['nomb_dist', 'nomb_prov', 'cod_dist']